# 1. Prepare the data (CPU notebook)

The training notebook reads a **cache** (`.npy` files, ~4 MB per patch) from a Kaggle dataset.
There are two ways to make it:

**A. Build it on another machine and upload it (recommended).** On a Linux machine with the
data extracted (for example the RTX A4000 box):
```
python scripts/prepare_sen12mscr.py cache --config configs/full.yaml --set data.root=/data/sen12ms_cr data.cache=/data/cache
pip install kaggle   # API token in ~/.kaggle/kaggle.json (Kaggle > Settings > API)
python kaggle/kaggle_tools.py publish --dir /data/cache --slug diewolke-cache
```
Then skip this notebook.

**B. Build it here.** This notebook streams the archives straight from TUM into extraction, so no
archive is ever stored. It extracts S1 first, uses its file names to pick the train/val/test
patches, and keeps only those patches from the two large S2 archives. Then it builds the cache in
`/kaggle/working/cache`. Save the version, and its output becomes the input of the training
notebook. Peak disk is about the extracted S1 data plus the chosen patches; the tool checks free
space before the S2 archives and stops early if they won't fit.

- Settings: Accelerator **None**, Internet **On**.
- Add the secret `SEN12MSCR_URL` (Add-ons > Secrets) and tick it for this notebook. The value is
  `https://dataserv.ub.tum.de/public.php/dav/files/m1554803` (TUM's public share; tested from
  Kaggle). FTP does not work from Kaggle.
- `/kaggle/working` is saved with the version, and users report a ~20 GB limit for it. So the
  default below caches 2,500 training and 1,500 test patches (about 17 GB). If you need the full
  test split, run this notebook a second time with `--splits test` only, and attach both outputs.

In [ ]:
import os, glob, shutil, time
T0 = time.time()                      # session start, used to stop training before Kaggle's 12 h limit
REPO_URL = ''                         # optional: git URL of the DieWolke repo, if it is pushed somewhere
CODE = '/kaggle/working/DieWolke'
if not os.path.exists(CODE):
    if REPO_URL:
        !git clone -q {REPO_URL} {CODE}
    else:  # the code uploaded as a Kaggle dataset (the DieWolke folder, without data or runs)
        hits = sorted(glob.glob('/kaggle/input/**/diewolke/__init__.py', recursive=True), key=len)
        assert hits, 'Add the DieWolke code as an input dataset (Add Input > Your Datasets), or set REPO_URL'
        shutil.copytree(os.path.dirname(os.path.dirname(hits[0])), CODE,
                        ignore=shutil.ignore_patterns('*.npy', 'runs', '.git', '.venv', '__pycache__'))
%cd {CODE}
!python kaggle/kaggle_tools.py env

In [ ]:
!pip install -q rasterio s2cloudless
from kaggle_secrets import UserSecretsClient
os.environ['SEN12MSCR_URL'] = UserSecretsClient().get_secret('SEN12MSCR_URL')

Archive sizes (a one-byte request each; HEAD requests are refused by this server):

In [ ]:
import subprocess
for f in ['ROIs1158_spring_s1.tar.gz', 'ROIs1158_spring_s2.tar.gz', 'ROIs1158_spring_s2_cloudy.tar.gz']:
    r = subprocess.run(['curl', '-sS', '-r', '0-0', '-o', '/dev/null', '-D', '-', '--max-time', '20',
                        os.environ['SEN12MSCR_URL'] + '/' + f], capture_output=True, text=True)
    rng = [l for l in r.stdout.splitlines() if l.lower().startswith('content-range')]
    print(f, f"{int(rng[-1].split('/')[-1]) / 1e9:.1f} GB" if rng else r.stdout.splitlines()[:1])

In [ ]:
!python kaggle/kaggle_tools.py prepare --out /kaggle/working/cache --n_train 2500 --n_test 1500 --delete_tifs
!ls -la /kaggle/working/cache | head -30

Now **Save Version > Save & Run All** (or Quick Save if it already ran). In the training
notebook, use **Add Input > Notebook Output** and pick this notebook. The tools find the cache
automatically.